In [ ]:
!pip install scikit-learn

In [ ]:
import kagglehub


path = kagglehub.dataset_download("hijest/genre-classification-dataset-imdb")

print("Path to dataset files:", path)

100%|██████████| 41.7M/41.7M [00:00<00:00, 55.9MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/hijest/genre-classification-dataset-imdb/versions/1


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r "/root/.cache/kagglehub/datasets/hijest/genre-classification-dataset-imdb/versions/1" "/content/drive/MyDrive/imdb_dataset"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

os.listdir("/content/drive/MyDrive/imdb_dataset/Genre Classification Dataset")


['description.txt',
 'test_data.txt',
 'test_data_solution.txt',
 'train_data.txt']

In [ ]:
file_content=[]

for file_name in os.listdir("/content/drive/MyDrive/imdb_dataset/Genre Classification Dataset"):
  with open("/content/drive/MyDrive/imdb_dataset/Genre Classification Dataset/"+file_name, 'r') as file:
    file_content.append(file.readlines())

In [ ]:
def split_text(row):
  splitted_text=row["Fulltext"].split(":::")
  columnn_names=["ID","Title","Genre","Description"] if len(splitted_text)==4 else ["ID","Title","Description"]

  return pd.Series(splitted_text, index=columnn_names)

In [ ]:
import pandas as pd

def convert_text_dataframe(file_content):
  df=pd.DataFrame(file_content, columns=["Fulltext"])
  df=df.join(df.apply(split_text, axis=1))
  df=df.drop(["Fulltext","ID"],axis=1)

  df["X Input"]=df["Title"]+" "+df["Description"]
  df.drop(["Title","Description"],axis=1,inplace=True)
  return df

In [ ]:
train_df=convert_text_dataframe(file_content[-1])
test_df=convert_text_dataframe(file_content[-2])

In [ ]:
train_df.head()

,Genre,X Input
0,drama,Oscar et la dame rose (2009) Listening in t...
1,thriller,Cupid (1997) A brother and sister with a pa...
2,adult,"Young, Wild and Wonderful (1980) As the bus..."
3,drama,The Secret Sin (1915) To help their unemplo...
4,drama,The Unrecovered (2007) The film's title ref...


In [ ]:
train_df.apply(lambda x: x.isna().sum())

,0
Genre,0
X Input,0


In [ ]:
test_df.apply(lambda x: x.isna().sum())

,0
Genre,0
X Input,0


In [ ]:
train_df["Genre"].value_counts()

,count
Genre,
drama,13613
documentary,13096
comedy,7447
short,5073
horror,2204
thriller,1591
action,1315
western,1032
reality-tv,884


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer=TfidfVectorizer(stop_words="english")
X=vectorizer.fit_transform(train_df["X Input"])


In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder=LabelEncoder()
y=encoder.fit_transform(train_df["Genre"])

In [ ]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(X,y,random_state=42)

In [ ]:
from sklearn.svm import LinearSVC

model=LinearSVC()
model.fit(x_train,y_train)


/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


LinearSVC()

In [ ]:
y_train_result=model.predict(x_train)
y_test_result=model.predict(x_test)



In [ ]:
from sklearn.metrics import accuracy_score,f1_score

traing_accuracy=accuracy_score(y_train,y_train_result)
print(f1_score(y_train,y_train_result,average="macro"))
print("Training Accuracy:",traing_accuracy)

0.9999563475958203
Training Accuracy: 0.9999508116084604


In [ ]:
testing_accuracy=accuracy_score(y_test,y_test_result)
print(f1_score(y_test,y_test_result,average="macro"))
print("Validation Accuracy:",testing_accuracy)

0.2539550758392082
Validation Accuracy: 0.4738822487826472


In [ ]:
X_test=vectorizer.transform(test_df["X Input"])
y_test=model.predict(X_test)

y_true=encoder.transform(test_df["Genre"])

In [ ]:
testing_accuracy=accuracy_score(y_true,y_test)
print(f1_score(y_true,y_test,average="macro"))
print("Test Accuracy:",testing_accuracy)

0.16199560258115717
Test Accuracy: 0.519889298892989
